# Corrosion Segmentation: Model Evaluation and Crack Analysis

This notebook loads a trained segmentation model, evaluates it on the validation set, and performs crack morphology analysis including length measurements.

In [ ]:
import os
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from .scripts.rustNet  import RustNet
import segmentation_models_pytorch as smp
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.measure import label
from skimage.morphology import skeletonize
from scipy.ndimage import convolve
from pytorch_grad_cam import AblationCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
# Configuration
CHECKPOINT_PATH = "./checkpoints/20241111_172413 Unet+resnet50+transform/best_model.pth"
IMAGES_DIR = "./Dataset/train/imgs"
MASKS_DIR = "./Dataset/train/masks"
PIXEL_SIZE_MM = 0.03  # Pixel size in millimeters for length conversion
THRESHOLD = 0.9  # Prediction threshold

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Model Loading

In [ ]:
# Initialize model: RustNet with ResNet50 encoder
model = RustNet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(device)

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Print checkpoint info
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"  Dice Score: {checkpoint['dice_score']:.4f}")
print(f"  IoU Score: {checkpoint['iou_score']:.4f}")

model.eval()

In [ ]:
# Initialize model: U-Net with ResNet50 encoder
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(device)

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Print checkpoint info
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"  Dice Score: {checkpoint['dice_score']:.4f}")
print(f"  IoU Score: {checkpoint['iou_score']:.4f}")

model.eval()

## Dataset

In [ ]:
class CustomDataset(Dataset):
    """Dataset for loading images and masks."""
    
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])
        
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask)

        return image, mask

In [ ]:
# Create dataset and dataloaders
transform = transforms.Compose([transforms.ToTensor()])

dataset = CustomDataset(images_dir=IMAGES_DIR, masks_dir=MASKS_DIR, transform=transform)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f"Train size: {train_size}, Validation size: {val_size}")

## Evaluation

In [ ]:
def evaluate_model(model, dataloader, device, threshold=0.5):
    """Evaluate model and compute Dice and IoU scores."""
    model.eval()
    dice_score, iou_score = 0.0, 0.0
    num_batches = len(dataloader)
    
    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = (outputs > threshold).float()
            
            intersection = (preds * masks).sum()
            dice = (2. * intersection) / (preds.sum() + masks.sum() + 1e-8)
            iou = intersection / (preds.sum() + masks.sum() - intersection + 1e-8)
            
            dice_score += dice.item()
            iou_score += iou.item()

    avg_dice = dice_score / num_batches
    avg_iou = iou_score / num_batches
    
    print(f"Average Dice Score: {avg_dice:.4f}")
    print(f"Average IoU Score: {avg_iou:.4f}")
    
    return avg_dice, avg_iou

# Run evaluation
avg_dice, avg_iou = evaluate_model(model, val_loader, device, threshold=THRESHOLD)

## Prediction Visualization

In [ ]:
# Visualize predictions with bounding boxes
with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(val_loader):
        images = images.to(device)
        outputs = model(images)
        preds = (outputs > THRESHOLD).float()
        
        images_np = images.cpu().numpy()
        preds_np = preds.cpu().numpy()
        masks_np = masks.cpu().numpy()
        
        for i in range(images_np.shape[0]):
            image = images_np[i].transpose(1, 2, 0)
            pred_mask = preds_np[i, 0]
            true_mask = masks_np[i, 0]
            
            # Normalize image
            image = (image - image.min()) / (image.max() - image.min() + 1e-8)
            
            # Create overlay
            overlay = image.copy()
            overlay[..., 0] = np.clip(overlay[..., 0] + pred_mask * 0.7, 0, 1)
            
            # Find and draw bounding boxes
            pred_mask_uint8 = (pred_mask * 255).astype(np.uint8)
            contours, _ = cv2.findContours(pred_mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if len(contours) > 0:
                overlay_uint8 = (overlay * 255).astype(np.uint8)
                for cnt in contours:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cv2.rectangle(overlay_uint8, (x, y), (x + w, y + h), color=(0, 255, 0), thickness=2)
                overlay = overlay_uint8.astype(np.float32) / 255.0
            
            plt.figure(figsize=(8, 8))
            plt.imshow(overlay)
            plt.title(f"Batch {batch_idx}, Image {i} - Predicted Segmentation")
            plt.axis('off')
            plt.show()

## Crack Morphology Analysis

In [ ]:
def find_endpoints(skeleton):
    """Find endpoints in a skeleton image."""
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]])
    neighbor_count = convolve(skeleton.astype(np.uint8), kernel, mode='constant', cval=0)
    endpoints = (neighbor_count == 11)
    return endpoints


def skeleton_to_graph(skeleton):
    """Build a graph from skeleton image."""
    G = nx.Graph()
    foreground_pixels = np.column_stack(np.where(skeleton))
    for y, x in foreground_pixels:
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                if dy == 0 and dx == 0:
                    continue
                ny, nx_ = y + dy, x + dx
                if 0 <= ny < skeleton.shape[0] and 0 <= nx_ < skeleton.shape[1]:
                    if skeleton[ny, nx_]:
                        G.add_edge((y, x), (ny, nx_), weight=np.hypot(dy, dx))
    return G


def extract_branches(G):
    """Extract individual branches from the skeleton graph."""
    nodes_degree = dict(G.degree())
    key_nodes = [node for node, degree in nodes_degree.items() if degree != 2]

    visited_edges = set()
    branches = []

    for node in key_nodes:
        for neighbor in G.neighbors(node):
            edge = (node, neighbor) if node <= neighbor else (neighbor, node)
            if edge in visited_edges:
                continue

            branch = [node, neighbor]
            visited_edges.add(edge)
            prev_node = node
            current_node = neighbor

            while True:
                if nodes_degree[current_node] != 2:
                    break

                neighbors = list(G.neighbors(current_node))
                neighbors.remove(prev_node)
                next_node = neighbors[0]

                edge = (current_node, next_node) if current_node <= next_node else (next_node, current_node)
                if edge in visited_edges:
                    break

                branch.append(next_node)
                visited_edges.add(edge)
                prev_node, current_node = current_node, next_node

            branches.append(branch)

    return branches

In [ ]:
# Analyze crack branches and measure lengths
with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(val_loader):
        images = images.to(device)
        outputs = model(images)
        preds = (outputs > THRESHOLD).float()

        images_np = images.cpu().numpy()
        preds_np = preds.cpu().numpy()

        for i in range(images_np.shape[0]):
            image = images_np[i].transpose(1, 2, 0)
            pred_mask = preds_np[i, 0]
            image = (image - image.min()) / (image.max() - image.min() + 1e-8)

            label_image = label(pred_mask)
            num_labels = label_image.max()
            print(f"Image {i}: Number of segments: {num_labels}")

            plt.figure(figsize=(10, 10))
            plt.imshow(image, cmap='gray')
            plt.title(f"Image {i} - Crack Branches")
            plt.axis('off')

            for label_value in range(1, num_labels + 1):
                component_mask = (label_image == label_value)
                skeleton = skeletonize(component_mask)
                G = skeleton_to_graph(skeleton)
                branches = extract_branches(G)
                print(f"  Segment {label_value}: {len(branches)} branches")

                for idx, branch_nodes in enumerate(branches):
                    path_length = 0.0
                    for j in range(len(branch_nodes) - 1):
                        node1 = branch_nodes[j]
                        node2 = branch_nodes[j + 1]
                        edge_data = G.get_edge_data(node1, node2)
                        path_length += edge_data['weight']

                    path_length_mm = path_length * PIXEL_SIZE_MM
                    start_point = branch_nodes[0]
                    end_point = branch_nodes[-1]

                    if path_length_mm > 1.0:
                        print(f"    Branch {idx + 1}: {path_length_mm:.2f} mm")
                        branch_coords = np.array(branch_nodes)
                        plt.plot(branch_coords[:, 1], branch_coords[:, 0], 'b-')
                        plt.scatter([start_point[1], end_point[1]], [start_point[0], end_point[0]], c='blue')
                        plt.annotate(f'{path_length_mm:.2f} mm', 
                                     xy=(start_point[1], start_point[0]),
                                     fontsize=10, color='black')

            plt.show()

## Single Image Inference

In [ ]:
def load_image(image_path, size=(512, 512)):
    """Load and preprocess a single image."""
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
    ])
    image = Image.open(image_path).convert('RGB')
    print(f"Original image size: {image.size}")
    tensor = transform(image).unsqueeze(0)
    print(f"Tensor shape: {tensor.shape}")
    return tensor

In [ ]:
# Load and analyze a single image
# Update this path to your test image
image_path = './pics/1516.jpg'
pixel_size_mm = 0.0335

images = load_image(image_path).to(device)
outputs = model(images)
preds = (outputs > 0.8).float()

images_np = images.cpu().numpy()
preds_np = preds.cpu().numpy()

for i in range(images_np.shape[0]):
    image = images_np[i].transpose(1, 2, 0)
    pred_mask = preds_np[i, 0]
    image = (image - image.min()) / (image.max() - image.min() + 1e-8)

    label_image = label(pred_mask)
    num_labels = label_image.max()
    print(f"Number of segments: {num_labels}")

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.title("Crack Analysis")
    plt.axis('off')

    for label_value in range(1, num_labels + 1):
        component_mask = (label_image == label_value)
        skeleton = skeletonize(component_mask)
        G = skeleton_to_graph(skeleton)
        branches = extract_branches(G)

        for idx, branch_nodes in enumerate(branches):
            path_length = sum(
                G.get_edge_data(branch_nodes[j], branch_nodes[j + 1])['weight']
                for j in range(len(branch_nodes) - 1)
            )
            path_length_mm = path_length * pixel_size_mm
            
            if path_length_mm > 1.0:
                print(f"  Branch: {path_length_mm:.2f} mm")
                start_point = branch_nodes[0]
                end_point = branch_nodes[-1]
                branch_coords = np.array(branch_nodes)
                plt.plot(branch_coords[:, 1], branch_coords[:, 0], 'b-')
                plt.scatter([start_point[1], end_point[1]], [start_point[0], end_point[0]], c='blue')
                plt.annotate(f'{path_length_mm:.2f} mm',
                             xy=(start_point[1], start_point[0]),
                             fontsize=10, color='black')

    plt.show()

## Grad-CAM Visualization

In [ ]:
class SegmentationSumTarget:
    """Target for Grad-CAM on segmentation models."""
    def __call__(self, model_output):
        return model_output.sum()


# Configure Grad-CAM
target_layers = [model.encoder.layer4[-1].conv3]
targets = [SegmentationSumTarget()]

# Load image for Grad-CAM
input_tensor = load_image(image_path).to(device)

with torch.no_grad():
    outputs = model(input_tensor)
    preds = (outputs > 0.9).float()

with AblationCAM(model=model, target_layers=target_layers) as cam:
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

images_np = input_tensor.detach().cpu().numpy()
preds_np = preds.detach().cpu().numpy()

for i in range(images_np.shape[0]):
    image = images_np[i].transpose(1, 2, 0)
    image_norm = (image - image.min()) / (image.max() - image.min() + 1e-8)
    
    cam_image = show_cam_on_image(image_norm, grayscale_cam[i, :], use_rgb=True)

    plt.figure(figsize=(8, 8))
    plt.imshow(cam_image)
    plt.title("Grad-CAM Visualization")
    plt.axis('off')
    plt.show()

In [2]:
!python statistical_validation.py

Statistical Validation for Corrosion Segmentation


Traceback (most recent call last):
  File "d:\Side projects\Mubeen\Project\Project-2\pytorch\statistical_validation.py", line 1295, in <module>
    main()
  File "d:\Side projects\Mubeen\Project\Project-2\pytorch\statistical_validation.py", line 1242, in main
    model = load_model()
  File "d:\Side projects\Mubeen\Project\Project-2\pytorch\statistical_validation.py", line 60, in load_model
    model.load_state_dict(checkpoint['model_state_dict'])
  File "c:\Users\abas\miniconda3\envs\HAR\lib\site-packages\torch\nn\modules\module.py", line 2584, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for Unet:
	size mismatch for segmentation_head.0.weight: copying a param with shape torch.Size([1, 16, 3, 3]) from checkpoint, the shape in current model is torch.Size([2, 16, 3, 3]).
	size mismatch for segmentation_head.0.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([2]).
